# HeartTriage DSS — Google Colab Edition

_Smart Triage, Faster Decisions_

Self-contained notebook: no repo clone needed. Upload `cleve.mod` when prompted.


In [ ]:
# Cell 1: Setup
!pip install pandas numpy matplotlib seaborn plotly tabulate rich -q

In [ ]:
# Cell 2: Upload data
from google.colab import files
uploaded = files.upload()  # upload cleve.mod
DATA_PATH = list(uploaded.keys())[0]

In [ ]:
# Cell 3: Configuration
COLUMNS = ['age','sex','cp','trestbps','chol','fbs','restecg',
           'thalach','exang','oldpeak','slope','ca','thal','class','subclass']

CRITERIA = ['age','trestbps','chol','thalach','oldpeak','ca','thal','exang']
BENEFIT = ['thalach']
WEIGHTS = {'age':0.10,'trestbps':0.15,'chol':0.10,'thalach':0.15,
           'oldpeak':0.15,'ca':0.15,'thal':0.10,'exang':0.10}
THRESHOLDS = {'P1':0.65,'P2':0.45}
MAPPING = {'sex':{'male':1,'fem':0},'cp':{'angina':1,'abnang':2,'notang':3,'asympt':4},
           'fbs':{'true':1,'fal':0},'restecg':{'norm':0,'abn':1,'hyp':2},
           'exang':{'true':1,'fal':0},'slope':{'up':1,'flat':2,'down':3},
           'thal':{'norm':3,'fixed':6,'fix':6,'rev':7}}

In [ ]:
# Cell 4: Load & clean
import numpy as np
import pandas as pd

df = pd.read_csv(DATA_PATH, sep=r'\s+', header=None, names=COLUMNS, comment='%')
df.replace('?', np.nan, inplace=True)
num_cols = ['age','trestbps','chol','thalach','oldpeak','ca']
df[num_cols] = df[num_cols].apply(pd.to_numeric, errors='coerce')
df.dropna(inplace=True)
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)
print(f'Clean: {df.shape[0]} rows')
df.head()

In [ ]:
# Cell 5: Transform categoricals
for col, mapping in MAPPING.items():
    df[col] = df[col].map(mapping)
df.head()

In [ ]:
# Cell 6: SAW normalization & scoring
norm = df[CRITERIA].copy()
for c in BENEFIT:
    norm[c] = df[c] / df[c].max()
for c in CRITERIA:
    if c not in BENEFIT:
        cmin = df[c].min()
        norm[c] = np.where(df[c] == cmin, 1.0, cmin / df[c])

df['score'] = sum(norm[c] * WEIGHTS[c] for c in norm.columns).round(4)
df[['age','score']].describe()

In [ ]:
# Cell 7: Triage categorization
def kategori(s):
    if s >= THRESHOLDS['P1']: return 'P1 - Emergency'
    elif s >= THRESHOLDS['P2']: return 'P2 - Urgent'
    else: return 'P3 - Non-Urgent'

df['triage'] = df['score'].apply(kategori)
dist = df['triage'].value_counts().reset_index()
dist.columns = ['triage','count']
dist['pct'] = (dist['count']/dist['count'].sum()*100).round(1)
dist

In [ ]:
# Cell 8: Visualization
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.countplot(x='triage', data=df, hue='triage', palette='Reds', legend=False, ax=axes[0])
axes[0].set_title('Triage Distribution')
sns.boxplot(x='triage', y='score', data=df, hue='triage', palette='Set2', legend=False, ax=axes[1])
axes[1].set_title('Score per Triage')
plt.tight_layout()
plt.show()

## Results

| Triage | Meaning |
|--------|---------|
| **P1** | Emergency — immediate treatment (< 5 min) |
| **P2** | Urgent — fast treatment (< 30 min) |
| **P3** | Non-Urgent — can wait (< 60 min) |
